In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
%matplotlib inline

In [ ]:
df_train = pd.read_csv('train_dataset_final1.csv')

In [ ]:
df_test = pd.read_csv('validate_dataset_final.csv')

In [ ]:
df_train.head()

In [ ]:
df_train.columns

In [ ]:
df_train.info()

Observation:

All the datatypes are either int or float. None are object meaning all categorical variables have already been handled and don't need to be handled before giving the data to the model.

In [ ]:
df_train.describe()

In [ ]:
df_test.head()

Observation:

The test data doesn't contain the column next_month_default as we have to predict it through our model. We can also see that the dataset of customer_id is completely useless as it won't be required for training the model so we can drop it. And for doing the preprocessing of the data we should first combine the training and test dataset.

In [ ]:
df = pd.concat([df_train,df_test], ignore_index = True)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
##Checking for missing values
df.isnull().sum()

Observation:

Age contains multiple missing values. We can replace the missing values in age by either mean or mode of all other age values.


In [ ]:
df['age'].unique()

In [ ]:
df['age'].median()

In [ ]:
df['age'].fillna(df['age'].median(), inplace = True)

In [ ]:
df['age'].unique()


In [ ]:
df.isnull().sum()


In [ ]:
df_train.columns


In [ ]:
df_test.columns

In [ ]:
df.columns

In [ ]:
train_df = df[df['next_month_default'].notnull()].copy()
test_df = df[df['next_month_default'].isnull()].copy()

In [ ]:
train_df.drop(columns=['Customer_ID'], inplace = True)

In [ ]:
train_df['next_month_default'] = train_df['next_month_default'].astype(int)


In [ ]:
train_df.columns

In [ ]:
sns.countplot(x='next_month_default', data=train_df)
plt.title("Target Variable Distribution")
plt.xlabel("Default (1 = Yes, 0 = No)")
plt.ylabel("Number of Customers")
plt.show()

Observation:

We can see here from the above plot that more than 80% of the customers in the dataset are non-defaulters whereas only 20% are defaulters which means that the data is highlt imbalanced. And if the data is trained on such highly imbalanced dataset the model might learn to predict only the majority class so we need to fix it to make the dataset more balanced.

In [ ]:
sns.countplot(x='sex', hue=train_df['next_month_default'].astype(str), data=train_df)


In [ ]:
sns.countplot(x='marriage', hue=train_df['next_month_default'].astype(str), data=train_df)


In [ ]:
df.head()

In [ ]:
df['marriage'].unique()

In [ ]:
df.marriage.value_counts()

Observation:

We can see that there are some discrepancies in the dataset ragarding the marriage values as the marriage column also contains value 0 whereas according to the problem statement only values like 1 (single), 2(married) and 3(Others) are allowed. Instead of dropping the rows which contain the incosistencies which could lead to data loss, what we will do is replace all the inconsistent values (0) with the mode of all the other values. This is logical because a very small percentage of data is incosistent. Out of over 30,000 rows only 62 contain incosistent values. So what we will do is first combine the training and test dataset and then handle these inconsistent values. But first let's check if there are inconsistencies in any other column values.

In [ ]:
df['sex'].unique()

In [ ]:
df['education'].unique()

Observation:
Even the education column contains inconsistent values like 0,5,6 but they are also in very less percentage so we can replace them with the mode value.

In [ ]:
df = pd.concat([train_df, test_df], ignore_index=True )

In [ ]:
df['education'].mode()[0]

In [ ]:
df['education'] = df['education'].map({5:2,6:2,0:2,1:1,2:2,3:3,4:4})

In [ ]:
df['education'].unique()

In [ ]:
df['marriage'].mode()[0]

In [ ]:
df['marriage'] = df['marriage'].map({2:2,3:3,1:1,0:2})

In [ ]:
##All inconsistent values in the dataset have been handled.

In [ ]:
train_df = df[df['next_month_default'].notnull()].copy()
test_df = df[df['next_month_default'].isnull()].copy()

In [ ]:
train_df.drop(columns=['Customer_ID'], inplace = True)

In [ ]:
sns.countplot(x='education', hue=train_df['next_month_default'].astype(str), data=train_df)


In [ ]:
train_df['age_group'] = pd.cut(train_df['age'], bins=[20,30,40,50,60,70,80], labels=["20–30", "31–40", "41–50", "51–60", "61–70", "71–80"])

sns.countplot(x='age_group', hue=train_df['next_month_default'].astype(str), data=train_df)
plt.title("Default Rate by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Count")
plt.legend(title='Default')
plt.show()

In [ ]:
train_df['credit_util'] = train_df['AVG_Bill_amt'] / (train_df['LIMIT_BAL'] + 1)  # Avoid divide by zero

sns.boxplot(x='next_month_default', y='credit_util', data=train_df)
plt.title("Credit Utilization vs Default")
plt.xlabel("Default")
plt.ylabel("Utilization Ratio")
plt.show()

Observation:
Credit utilization indicates how much of their credit limit is the customer using on average. Higher utilization usually indicates higher risk of default. So credit utlization may be a strong financial signal indicating whether the customer may default or not. The box plot clearly shows that median for both the boxes lies between 0.3 and 0.5. We can see thatdefaulters have a slightly higher median and onger upper whisker indicating outliers having very high credit utilization meaning that customers that customers that default tend to have slightly higher credit utilization ratio on avaerage.


In [ ]:
pay_amt_cols = ['pay_amt1', 'pay_amt2', 'pay_amt3', 'pay_amt4', 'pay_amt5', 'pay_amt6']
train_df['total_pay_amt'] = train_df[pay_amt_cols].sum(axis=1)

sns.boxplot(x='next_month_default', y='total_pay_amt', data=train_df)
plt.title("Total Repayment Amount vs Default")
plt.xlabel("Default")
plt.ylabel("Total Payment (Last 6 Months)")
plt.yscale('log')  
plt.show()


Higher totel_pay_amount indicates that the customer is serious about paying back the dues and keeps him at low risk of dedaulting the next month.

In [ ]:
train_df['repayment_ratio'] = train_df['total_pay_amt'] / (train_df['AVG_Bill_amt'] * 6 + 1)

sns.boxplot(x='next_month_default', y='repayment_ratio', data=train_df)
plt.title("Repayment Ratio vs Default")
plt.xlabel("Default")
plt.ylabel("Repayment / Bill Amount Ratio")
plt.show()

Observation:
The box plot looks like this because most of the ratios are near to zero and hence the median comes very close to zero.

In [ ]:
train_df.columns

In [ ]:
pay_cols = ['pay_0', 'pay_2', 'pay_3', 'pay_4', 'pay_5', 'pay_6']
train_df['num_delays'] = (train_df[pay_cols] > 0).sum(axis=1)

sns.histplot(data=train_df, x='num_delays', hue='next_month_default', multiple='stack', bins=7)
plt.title("Number of Months with Payment Delays")
plt.xlabel("Count of Delayed Months")
plt.ylabel("Customers")
plt.legend(title="Default")
plt.show()



Observation:
We can see here that as the number of months having delayed payments increases the ratio of defaulters to non-defaulters also increases. We can also see that when the number of months with delayed payments reaches 4 and above the number of customers that default are more than customers that don't.

In [ ]:
overdue_count = 0

# Start from i = 2 to 6 since we need PAY_AMT[i] vs BILL_AMT[i-1]
for i in range(2, 7):
    bill_col = f'Bill_amt{i-1}'
    pay_col = f'pay_amt{i}'
    overdue = train_df[bill_col] > train_df[pay_col]
    overdue_count += overdue.astype(int)

# Add to DataFrame
train_df['overdue_count'] = overdue_count


Here we have engineered another feature overdue_count which tells us for how many months there was overdue payment meaning the bill amount was more than the payment amount. It tells us if the person has a habit of making partial or less payments.

In [ ]:
train_df.info(0)

In [ ]:
bill_amt_cols = ['Bill_amt1', 'Bill_amt2', 'Bill_amt3', 'Bill_amt4', 'Bill_amt5', 'Bill_amt6']
train_df['total_bill_amt'] = train_df[bill_amt_cols].sum(axis=1)

In [ ]:
train_df.drop(columns=['Bill_amt1', 'Bill_amt2', 'Bill_amt3', 'Bill_amt4', 'Bill_amt5', 'Bill_amt6', 'pay_amt1', 'pay_amt2', 'pay_amt3', 'pay_amt4', 'pay_amt5', 'pay_amt6'], inplace=True)

Since we have already engineered features like total_pay_amount and total_bill_amount it feels redundant to keep the individual values of payments and bills for each of the month so we can just drop them.

In [ ]:
train_df.info()

Observation:
Let's first engineer some features and transformations that are financially meaningful. We have already engineered one which is credit_utilization. Let's engineer one more which is deliquency_streak which means the the number of recent consecutive 

In [ ]:
def delinquency_streak(row):
    streak = 0
    max_streak = 0
    for val in row:
        if val > 0:
            streak += 1
            max_streak = max(max_streak, streak)
        else:
            streak = 0
    return max_streak

train_df['delinq_streak'] = train_df[pay_cols].apply(delinquency_streak, axis=1)

Observation:
We have to drop all categorical variables that we created during EDA before doing SMOTE.

In [ ]:
train_df.drop(columns=[ 'age_group'], inplace=True)

So now for handling the highly imbalanced data we will perform SMOTE on the data. This helps the model learn from a more balanced dataset, improving its ability to detect rare default cases.

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

X = train_df.drop(columns=['next_month_default']) 
y = train_df['next_month_default']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

from collections import Counter
print(Counter(y_train_smote))


In [ ]:
##Using logistic regression model
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve
import numpy as np


logreg = LogisticRegression(max_iter=1000, class_weight=None, random_state=42)
logreg.fit(X_train_smote, y_train_smote)


y_probs = logreg.predict_proba(X_test)[:, 1]  


precision, recall, thresholds = precision_recall_curve(y_test, y_probs)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-6)
best_threshold = thresholds[np.argmax(f1_scores)]


y_pred_thresh = (y_probs >= best_threshold).astype(int)


print("===== Logistic Regression =====")
print(f"Best Threshold: {best_threshold:.2f}")
print(confusion_matrix(y_test, y_pred_thresh))
print(classification_report(y_test, y_pred_thresh))
print(f"AUC-ROC: {roc_auc_score(y_test, y_probs):.6f}")


In [ ]:
##Using Decision Trees
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve
import numpy as np


dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train_smote, y_train_smote)


y_proba = dt_model.predict_proba(X_test)[:, 1]  #

# Step 3: Find optimal threshold (maximize F1-score or balance precision/recall)
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-6)  # avoid division by zero
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
print(f"\nBest Threshold: {best_threshold:.2f}")

# Step 4: Predict using optimal threshold
y_pred_thresh = (y_proba >= best_threshold).astype(int)

# Step 5: Evaluate the model
print("\n===== Decision Tree Evaluation =====")
print(confusion_matrix(y_test, y_pred_thresh))
print(classification_report(y_test, y_pred_thresh))
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}")


In [ ]:
##Using Ensemble Methods like XgBoost
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve
import numpy as np

# Step 1: Train the XGBoost model
xgb_model = XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)
xgb_model.fit(X_train_smote, y_train_smote)

# Step 2: Predict probabilities
y_proba = xgb_model.predict_proba(X_test)[:, 1]  # probability of class 1

# Step 3: Find optimal threshold using F1-score
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-6)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
print(f"\nBest Threshold: {best_threshold:.2f}")

# Step 4: Predict using optimal threshold
y_pred_thresh = (y_proba >= best_threshold).astype(int)

# Step 5: Evaluate the model
print("\n===== XGBoost Evaluation =====")
print(confusion_matrix(y_test, y_pred_thresh))
print(classification_report(y_test, y_pred_thresh))
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}")


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt


xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_smote, y_train_smote)


y_pred_xgb = xgb_model.predict(X_test)
y_probs_xgb = xgb_model.predict_proba(X_test)[:, 1]


print("===== XGBoost Evaluation =====")
print(confusion_matrix(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))
print(f"AUC-ROC Score: {roc_auc_score(y_test, y_probs_xgb):.4f}")


disp = ConfusionMatrixDisplay.from_estimator(xgb_model, X_test, y_test, display_labels=["No Default", "Default"], cmap="Blues")
disp.ax_.set_title("XGBoost Confusion Matrix")
plt.show()

XGBoost is the best model overall. It has the highest AUC-ROC (0.7467), indicating superior class separation, and its accuracy (0.79) and F1-scores (macro: 0.69, weighted: 0.79) are very close to the Decision Tree's, which only slightly edges out in accuracy and weighted F1-score. So we will use XGBoost model to make predictions on the test dataset. Now let's apply the model on our test dataset. We have the test dataset as test_df which has been preprocessed along with the training dataset for columns like age, education,marriage etc. We just need to add the new features that we added to the training dataset.

In [ ]:
test_df['credit_util'] = test_df['AVG_Bill_amt'] / (test_df['LIMIT_BAL'] + 1)  
pay_amt_cols = ['pay_amt1', 'pay_amt2', 'pay_amt3', 'pay_amt4', 'pay_amt5', 'pay_amt6']
test_df['total_pay_amt'] = test_df[pay_amt_cols].sum(axis=1)
test_df['repayment_ratio'] = test_df['total_pay_amt'] / (test_df['AVG_Bill_amt'] * 6 + 1)
test_df['num_delays'] = (test_df[pay_cols] > 0).sum(axis=1)

overdue_count = 0

# Start from i = 2 to 6 since we need PAY_AMT[i] vs BILL_AMT[i-1]
for i in range(2, 7):
    bill_col = f'Bill_amt{i-1}'
    pay_col = f'pay_amt{i}'
    overdue = test_df[bill_col] > test_df[pay_col]
    overdue_count += overdue.astype(int)


test_df['overdue_count'] = overdue_count

bill_amt_cols = ['Bill_amt1', 'Bill_amt2', 'Bill_amt3', 'Bill_amt4', 'Bill_amt5', 'Bill_amt6']
test_df['total_bill_amt'] = test_df[bill_amt_cols].sum(axis=1)

test_df.drop(columns=['Bill_amt1', 'Bill_amt2', 'Bill_amt3', 'Bill_amt4', 'Bill_amt5', 'Bill_amt6', 'pay_amt1', 'pay_amt2', 'pay_amt3', 'pay_amt4', 'pay_amt5', 'pay_amt6'], inplace=True)\

test_df['delinq_streak'] = test_df[pay_cols].apply(delinquency_streak, axis=1)\




In [ ]:
test_df.columns

In [ ]:
train_df.columns

In [ ]:

test_features = test_df.drop(columns=['Customer_ID', 'next_month_default'], errors='ignore')


validation_probs = xgb_model.predict_proba(test_features)[:, 1]


best_threshold = 0.36
validation_preds = (validation_probs >= best_threshold).astype(int)


final_output = test_df[['Customer_ID']].copy()
final_output['next_month_default'] = validation_preds


final_output.to_csv("submission_22124001.csv", index=False)

print(" Exported: submission_22124001.csv with Customer_ID and predicted default status.")
